# Page 7: Frequency Voting

Qwen2.5-3B-Instruct answers 100 MATH-500 problems many times over, and the answer most chains agree on is kept. 16 chains per problem are generated at each of 13 temperatures, 0.0 to 1.2, and the vote is scored at each one.

Built for a Colab A100. Every set of chains is saved to Google Drive as it is generated, so a run that disconnects picks up where it stopped when the cells are run again.

### Mount Google Drive for Saving Results
Run this cell to connect your Google Drive. All chains and tables are saved to a folder named `Page7_Frequency_Voting` in your Drive. Outside Colab they go to a local folder of the same name.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/Page7_Frequency_Voting'
except ImportError:
    SAVE_DIR = 'Page7_Frequency_Voting'

os.makedirs(SAVE_DIR, exist_ok=True)
os.chdir(SAVE_DIR)
print(f"Working directory: {os.getcwd()}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Page7_Frequency_Voting


### Step 1: Install Dependencies, Imports and Configurations

Every number the run uses is set here.

In [ ]:
#@title Installation and Setup
!pip install -q datasets transformers accelerate math_verify

import json
import math
import random
import time
from collections import Counter
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from math_verify import parse, verify
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-3B-Instruct"
N_PROBLEMS = 100       # drawn from the 500, in proportion to their levels
N_CHAINS = 16          # chains generated per problem, per setting
TOP_P = 0.95
SEED = 42

TEMPERATURES = [round(0.1 * i, 1) for i in range(13)]   # 0.0 to 1.2; 0.0 is greedy decoding
CAP = 2048                                             # tokens per chain
VOTE_SIZES = [1, 2, 4, 8, 16]

N_DRAWS = 100                # random subsets of the 16 chains averaged per point
TOKENS_IN_FLIGHT = 640_000   # rows x tokens per generate call; lower it on a smaller GPU
MAX_ROWS = 256               # most chains generated in one call

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"temperatures: {TEMPERATURES}")

temperatures: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]


### Step 2: Load the Problems

We pull the `HuggingFaceH4/MATH-500` dataset and draw 100 of its problems at random, the same share from each level as in the full 500.

In [ ]:
print("Loading MATH-500 dataset...")
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")

# The same share of each level as the 500, drawn at random within each level, kept in
# dataset order.
by_level = {}
for k, row in enumerate(dataset):
    by_level.setdefault(row["level"], []).append(k)
draw = random.Random(SEED)
quota = {lv: round(len(ks) * N_PROBLEMS / len(dataset)) for lv, ks in by_level.items()}
quota[max(quota, key=lambda lv: len(by_level[lv]))] += N_PROBLEMS - sum(quota.values())
picked = sorted(k for lv, ks in by_level.items() for k in draw.sample(ks, quota[lv]))
problems = [dataset[k] for k in picked]
golds = [p["answer"] for p in problems]

full = Counter(row["level"] for row in dataset)
levels = Counter(p["level"] for p in problems)
print(f"{len(problems)} of {len(dataset)} problems, by level: "
      + ", ".join(f"{lv}: {levels[lv]} of {full[lv]}" for lv in sorted(levels)))

Loading MATH-500 dataset...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

100 of 500 problems, by level: 1: 9 of 43, 2: 18 of 90, 3: 21 of 105, 4: 26 of 128, 5: 26 of 134


### Step 3: Prompt, Answers and the Vote

- **Prompt**: the problem, then an instruction to put the final answer in `\boxed{}`. Qwen's default system prompt is kept.
- **A chain's answer** is the last `\boxed{}` it writes. A chain with no boxed answer casts no vote.
- **Two answers are the same** when `math_verify` says they are equal, so `\frac{1}{2}` and `0.5` count as one answer.
- **The vote** is the most common answer among the chains. A tie counts as wrong.

In [ ]:
BOX = "Put your final answer within \\boxed{}."


def build_prompt(problem):
    return [{"role": "user", "content": f"{problem}\n\n{BOX}"}]


def extract_answer(text):
    start = text.rfind("\\boxed{")
    if start == -1:
        return ""  # no boxed answer
    start += len("\\boxed{")
    depth = 1
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        if depth == 0:
            return text[start:i].strip()
    return ""  # braces never closed


def grade(pred, gold):
    gt = str(gold).strip()
    if pred == "":
        return False
    try:
        return bool(verify(parse(f"${gt}$"), parse(f"${pred}$")))
    except Exception:
        return pred.replace(" ", "").lower() == gt.replace(" ", "").lower()


def same_answer(a, b):
    if a.replace(" ", "") == b.replace(" ", ""):
        return True
    try:
        return bool(verify(parse(f"${a}$"), parse(f"${b}$")))
    except Exception:
        return False


def tally(chains, gold):
    # Groups the chains' answers into distinct answers. Returns each chain's group
    # (None when it has no answer) and whether each group is the right answer.
    groups, labels = [], []
    for chain in chains:
        answer = chain["answer"]
        if answer == "":
            labels.append(None)
            continue
        for g, rep in enumerate(groups):
            if same_answer(rep, answer):
                labels.append(g)
                break
        else:
            groups.append(answer)
            labels.append(len(groups) - 1)
    return labels, [grade(rep, gold) for rep in groups]


def vote_right(labels, right):
    # Majority vote over the chains given. A tie counts as wrong.
    top = Counter(l for l in labels if l is not None).most_common(2)
    if not top or (len(top) == 2 and top[0][1] == top[1][1]):
        return False
    return right[top[0][0]]


def chains_right(labels, right):
    return sum(l is not None and right[l] for l in labels)


def chain_sets(n, rng):
    # Which n of the 16 chains vote: every combination when there are at most
    # N_DRAWS of them, otherwise N_DRAWS random ones.
    if math.comb(N_CHAINS, n) <= N_DRAWS:
        return [list(c) for c in combinations(range(N_CHAINS), n)]
    return [sorted(rng.choice(N_CHAINS, n, replace=False).tolist()) for _ in range(N_DRAWS)]


# Three chains say 1/2 two different ways and one says 3: the vote is 1/2.
demo = [{"answer": "\\frac{1}{2}"}, {"answer": "0.5"}, {"answer": "3"}, {"answer": ""}]
labels, right = tally(demo, "\\frac12")
print("groups:", labels, " right:", right, " vote right:", vote_right(labels, right))
print("a 1-1 tie:", vote_right(labels[1:3], right))

groups: [0, 0, 1, None]  right: [True, False]  vote right: True
a 1-1 tie: False


### Step 4: Load the Model and the Chain Generator

`generate_chains(temperature, length)` makes 16 chains for every problem at one temperature, each capped at `length` new tokens, and saves them to `chains_T{temperature}_L{length}.json`. Chains already saved for a problem, in that file or in a run that stopped midway, are read instead of generated again. Problems are generated several at a time, as many as `TOKENS_IN_FLIGHT` allows at that length.

At temperature 0.0 the 16 chains would be identical, so one is generated and copied. The checkpoint ships `top_k=20` and `repetition_penalty=1.05`; both are turned off so temperature and top-p are the only sampling settings.

Each chain records its text, its answer, the tokens it generated, and whether it hit its cap.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
eos = model.generation_config.eos_token_id
end_ids = {tokenizer.pad_token_id, *(eos if isinstance(eos, list) else [eos])}


def read_chain(ids, length):
    # A chain ends at its first end-of-turn or padding token; the rest is batch padding.
    ids = ids.tolist()
    n = next((k for k, t in enumerate(ids) if t in end_ids), len(ids))
    text = tokenizer.decode(ids[:n], skip_special_tokens=True)
    return {"text": text, "answer": extract_answer(text), "tokens": n, "truncated": n >= length}


def generate_chains(temperature, length):
    # Returns one record per problem in `problems`, in order. A file can hold more
    # problems than this run uses (an earlier run of all 500); only these are returned.
    path = f"chains_T{temperature:.1f}_L{length}.json"
    partial = path + ".partial"          # progress so far, if a run stopped midway
    saved = {}
    for f_name in (path, partial):
        if os.path.exists(f_name):
            with open(f_name) as f:
                saved.update({r["unique_id"]: r for r in json.load(f)})
    todo = [p for p in problems if p["unique_id"] not in saved]
    if not todo:
        return [saved[p["unique_id"]] for p in problems]

    greedy = temperature == 0.0
    per_problem = 1 if greedy else N_CHAINS
    rows = min(MAX_ROWS, TOKENS_IN_FLIGHT // (length + 512))
    step = max(1, rows // per_problem)   # problems per generate call

    start = time.time()
    for i in range(0, len(todo), step):
        batch = todo[i : i + step]
        prompts = [tokenizer.apply_chat_template(build_prompt(p["problem"]), tokenize=False,
                                                 add_generation_prompt=True) for p in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        input_len = inputs.input_ids.shape[1]

        torch.manual_seed(SEED + i)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=length,
                do_sample=not greedy,
                temperature=None if greedy else temperature,
                top_p=None if greedy else TOP_P,
                top_k=None if greedy else 0,   # 0 turns top-k off
                repetition_penalty=1.0,
                num_return_sequences=per_problem,
            )

        for j, p in enumerate(batch):
            chains = [read_chain(out[j * per_problem + k, input_len:], length)
                      for k in range(per_problem)]
            saved[p["unique_id"]] = {"unique_id": p["unique_id"],
                                     "chains": chains * (N_CHAINS // per_problem)}
        with open(partial, "w") as f:
            json.dump(list(saved.values()), f)
        done = len(problems) - len(todo) + min(i + step, len(todo))
        print(f"    T={temperature:.1f}  cap {length}: {done}/{len(problems)} problems "
              f"({time.time() - start:.0f}s)")

    os.replace(partial, path)
    return [saved[p["unique_id"]] for p in problems]


def tallies_for(records):
    return [tally(r["chains"], g) for r, g in zip(records, golds)]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Step 5: The Temperature Sweep

16 chains per problem at each of the 13 temperatures, top-p 0.95, each chain capped at 2,048 tokens. Each set of 16 is scored by its vote. The temperature whose vote is right on the most problems is the best temperature; a tie goes to the lower temperature.

For the figures, the vote is also taken over 1, 2, 4 and 8 of the 16 chains: every combination of them when there are at most `N_DRAWS`, otherwise `N_DRAWS` random ones. Each set's total tokens is recorded in `temperature_sets.csv`, and every chain is listed in `chains.csv`.

In [ ]:
sweep, sweep_tally = {}, {}
for t in TEMPERATURES:
    print(f"temperature {t:.1f}")
    sweep[t] = generate_chains(t, CAP)
    sweep_tally[t] = tallies_for(sweep[t])

summary = pd.DataFrame([{
    "temperature": t,
    "vote_accuracy": np.mean([vote_right(l, r) for l, r in sweep_tally[t]]),
    "chain_accuracy": np.mean([chains_right(l, r) / N_CHAINS for l, r in sweep_tally[t]]),
    "distinct_answers": np.mean([len(r) for _, r in sweep_tally[t]]),
    "mean_tokens": np.mean([c["tokens"] for rec in sweep[t] for c in rec["chains"]]),
    "hit_cap": np.mean([c["truncated"] for rec in sweep[t] for c in rec["chains"]]),
} for t in TEMPERATURES])
summary.to_csv("temperature_summary.csv", index=False)

best_t = float(summary.loc[summary.vote_accuracy.idxmax(), "temperature"])
display(summary.round(3))
print(f"\nbest temperature: {best_t}  (vote accuracy {summary.vote_accuracy.max():.0%})")

rng = np.random.default_rng(SEED)
rows = []
for t in TEMPERATURES:
    for p, (rec, (labels, right)) in enumerate(zip(sweep[t], sweep_tally[t])):
        for n in VOTE_SIZES:
            for d, idx in enumerate(chain_sets(n, rng)):
                rows.append({
                    "temperature": t, "problem": p, "n_chains": n, "draw": d,
                    "total_tokens": sum(rec["chains"][i]["tokens"] for i in idx),
                    "vote_right": vote_right([labels[i] for i in idx], right),
                })
temperature_sets = pd.DataFrame(rows)
temperature_sets.to_csv("temperature_sets.csv", index=False)
print(f"{len(temperature_sets)} sets recorded")

pd.DataFrame([
    {"temperature": t, "cap": CAP, "problem": p, "chain": c, "tokens": ch["tokens"],
     "hit_cap": ch["truncated"], "answer": ch["answer"]}
    for t in TEMPERATURES
    for p, rec in enumerate(sweep[t])
    for c, ch in enumerate(rec["chains"])
]).to_csv("chains.csv", index=False)

temperature 0.0
temperature 0.1
temperature 0.2
temperature 0.3
temperature 0.4
temperature 0.5
temperature 0.6
temperature 0.7
temperature 0.8
temperature 0.9
temperature 1.0
    T=1.0  cap 2048: 75/100 problems (490s)
    T=1.0  cap 2048: 90/100 problems (827s)
    T=1.0  cap 2048: 100/100 problems (1149s)
temperature 1.1
    T=1.1  cap 2048: 15/100 problems (513s)
    T=1.1  cap 2048: 30/100 problems (969s)
    T=1.1  cap 2048: 45/100 problems (1500s)
    T=1.1  cap 2048: 60/100 problems (2023s)
    T=1.1  cap 2048: 75/100 problems (2506s)
    T=1.1  cap 2048: 90/100 problems (2790s)
    T=1.1  cap 2048: 100/100 problems (3158s)
temperature 1.2
    T=1.2  cap 2048: 15/100 problems (386s)
    T=1.2  cap 2048: 30/100 problems (867s)


### Step 6: Heatmap, Every Problem at Every Temperature

One square per problem and temperature. Green means the vote of 16 chains is right, blue means it is wrong or tied, and a darker square means more of the 16 chains are right. Problems are sorted by how many chains got them right across all temperatures.

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.ticker import FuncFormatter, NullFormatter, PercentFormatter

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "font.size": 11,
})

right_counts = np.array([[chains_right(l, r) for l, r in sweep_tally[t]] for t in TEMPERATURES])
vote_ok = np.array([[vote_right(l, r) for l, r in sweep_tally[t]] for t in TEMPERATURES])
order = np.argsort(-right_counts.sum(axis=0), kind="stable")

greens = ListedColormap(plt.cm.Greens(np.linspace(0.15, 1, 256)))
blues = ListedColormap(plt.cm.Blues(np.linspace(0.15, 1, 256)))
shade = right_counts[:, order] / N_CHAINS
image = np.where(vote_ok[:, order, None], greens(shade), blues(shade))

fig, ax = plt.subplots(figsize=(16, 4.8))
ax.imshow(image, aspect="auto", origin="lower", interpolation="nearest")
ax.set_yticks(np.arange(-0.5, len(TEMPERATURES)), minor=True)
ax.grid(which="minor", color="#fcfcfb", linewidth=0.6)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
ax.set_yticks(range(len(TEMPERATURES)), [f"{t:.1f}" for t in TEMPERATURES])
ax.set_xticks([])
ax.set_xlabel(f"{len(problems)} problems, most chains right on the left")
ax.set_ylabel("temperature")
ax.set_title("The vote of 16 chains, per problem and temperature", loc="left")
for x, cmap, label in [(0.0, greens, "vote right"), (0.35, blues, "vote wrong or tied")]:
    bar = fig.colorbar(ScalarMappable(Normalize(0, N_CHAINS), cmap), orientation="horizontal",
                       cax=ax.inset_axes([x, -0.3, 0.3, 0.05]))
    bar.set_label(f"{label}: chains right, of {N_CHAINS}")
    bar.outline.set_visible(False)
plt.savefig("fig_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 7: Accuracy Against the Number of Chains, by Temperature

Accuracy of the vote over 1, 2, 4, 8 and 16 chains, one line per temperature. Each point is averaged over the sets of chains from step 5. The 0.0 line is greedy decoding: its 16 chains are identical, so it stays flat.

In [ ]:
acc_by_t = temperature_sets.groupby(["temperature", "n_chains"]).vote_right.mean().unstack(0)
hot = [t for t in TEMPERATURES if t > 0]
shades = plt.cm.Blues(np.linspace(0.35, 1, len(hot)))

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(acc_by_t.index, acc_by_t[0.0], color="#0b0b0b", linestyle="--", marker="o", label="0.0 (greedy)")
for t, color in zip(hot, shades):
    ax.plot(acc_by_t.index, acc_by_t[t], color=color, marker="o", markersize=5,
            linewidth=3 if t == best_t else 1.5, label=f"{t:.1f}" + (" (best)" if t == best_t else ""))
ax.set_xscale("log", base=2)
ax.set_xticks(VOTE_SIZES, [str(n) for n in VOTE_SIZES])
ax.set_xlabel("chains in the vote")
ax.set_ylabel("accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Majority voting, one line per temperature", loc="left")
ax.legend(title="temperature", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.savefig("fig_accuracy_by_temperature.png", dpi=150, bbox_inches="tight")
plt.show()
display(acc_by_t.round(3))

### Step 8: How Many Different Answers Each Temperature Produces

The number of different answers among the 16 chains, averaged over the problems. A chain with no boxed answer adds no answer.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([f"{t:.1f}" for t in TEMPERATURES], summary.distinct_answers, color="#2a78d6", width=0.7)
ax.set_xlabel("temperature")
ax.set_ylabel("different answers among 16 chains")
ax.set_title("Different answers per problem, by temperature", loc="left")
ax.grid(axis="x", visible=False)
plt.savefig("fig_distinct_answers.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 9: Accuracy Against Tokens Spent

Majority voting at the best temperature over 1, 2, 4, 8 and 16 chains, against the tokens spent per problem: the total tokens of the chains used, averaged over the 100 problems. Greedy decoding is a single point.

In [ ]:
spend = (temperature_sets[temperature_sets.temperature == best_t]
         .groupby("n_chains").agg(tokens=("total_tokens", "mean"), accuracy=("vote_right", "mean")))
greedy = temperature_sets[(temperature_sets.temperature == 0.0) & (temperature_sets.n_chains == 1)]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(spend.tokens, spend.accuracy, color="#2a78d6", marker="o", markersize=8,
        label=f"majority vote, temperature {best_t}")
for n, row in spend.iterrows():
    ax.annotate(f"{n} chain" + ("s" if n > 1 else ""), (row.tokens, row.accuracy),
                xytext=(0, 10), textcoords="offset points", ha="center", color="#52514e")
ax.plot(greedy.total_tokens.mean(), greedy.vote_right.mean(), "D", color="#0b0b0b",
        markersize=9, label="greedy decoding")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("tokens spent per problem")
ax.set_ylabel("accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("What the vote costs", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.savefig("fig_accuracy_vs_tokens.png", dpi=150, bbox_inches="tight")
plt.show()

spend.loc["greedy"] = [greedy.total_tokens.mean(), greedy.vote_right.mean()]
display(spend.round(3))